In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import sys
sys.path.append("..")
from multiHIVE.model import multiHIVE
import torch

In [13]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
torch.set_float32_matmul_precision("high")

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
adata = sc.read_h5ad("../../Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata.var_names_make_unique()
del adata.obsm
del adata.obsp
adata

AnnData object with n_obs × n_vars = 25517 × 165454
    obs: 'cell_type', 'batch'
    var: 'modality'

In [ ]:
adata = scvi.data.organize_multiome_anndatas(adata)
adata = adata[:, adata.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata, min_cells=int(adata.shape[0] * 0.01))
multiHIVE.setup_anndata(adata, batch_key="modality")
adata

In [8]:
vae = multiHIVE(adata, latent_distribution="normal",
                n_genes=(adata.var["modality"] == "Gene Expression").sum(),
                n_regions=(adata.var["modality"] == "Peaks").sum(),
                n_proteins=0,
                # kl_dot_product  = False,
                # deep_network = True,
               )

Encoder(
  (encoder_r_1): Sequential(
    (0): Linear(in_features=48873, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (z_mean_encoder_delta_1): Linear(in_features=128, out_features=20, bias=True)
  (z_var_encoder_delta_1): Linear(in_features=128, out_features=20, bias=True)
  (encoder_r_2): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_sta

In [ ]:
vae.train()
vae.get_latent_representation()

In [10]:
dataset = "TEA-seq"
np.save(f"{dataset}.npy", adata.obsm['Z_multiHIVE'])

In [ ]:
def evaluate_clustering(labels_true, labels_pred):
    from sklearn import metrics

    results = {
        'normalized_mutual_info': metrics.normalized_mutual_info_score(labels_true, labels_pred),
        'adjusted_rand_index': metrics.adjusted_rand_score(labels_true, labels_pred),
        'fowlkes_mallows': metrics.fowlkes_mallows_score(labels_true, labels_pred),
    }
    return results

import scib
adata = sc.read_h5ad("../../Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata.obsm['latent'] = np.load(f"./{dataset}.npy")

results_bio = {}

sc.pp.neighbors(adata,use_rep="latent")
for res in np.linspace(0.1, 2, 20):
    sc.tl.leiden(adata, resolution = res, key_added="Z_leiden")
    n_clusters = np.unique(adata.obs['Z_leiden']).shape[0]
    y_pred = adata.obs['Z_leiden']
    result = evaluate_clustering(y_pred,adata.obs['cell_type'].values)
    results_bio[res] = result
results_bio = pd.DataFrame(results_bio).T
results_bio.to_csv(f'./{dataset}_metrics_bio.csv')

asw_val = scib.me.silhouette(adata, label_key="cell_type", embed="latent")
print("ASW", asw_val)

sc.pp.neighbors(adata, use_rep="latent")
gc_val = scib.me.graph_connectivity(adata, label_key="cell_type")
print("Graph_Connectivity", gc_val)

asw_batch_val = scib.me.silhouette_batch(adata, batch_key="batch", label_key="cell_type", embed="latent")
print("ASW_batch", asw_batch_val)

results_batch = pd.DataFrame([
            ["ASW", asw_val], 
            ["Graph Connectiviy", gc_val],
            ["ASW-batch", asw_batch_val]
        ]
    )
results_batch.to_csv(f'./{dataset}_metrics_batch.csv')